# Глубокие сети и длина последовательности

**Цель:** сравнить LSTM, GRU, 1D CNN и Transformer на одинаковых данных при разных длинах окна: **5, 15, 30, 60, 120, 180** баров.

- Данные и split: как в проде, temporal split **8 train / 1 val / 1 test**.
- Фичи: baseline из `features_selected_tp_sl_1_05.txt` (без rolling-дубликатов).
- Фреймворк: **PyTorch**, использование GPU при наличии CUDA (`device='cuda'`).

## Логика сделок (как в продовых экспериментах)

- Сигналы по вероятности:
  - `p >= threshold` -> **long**
  - `p <= 1 - threshold` -> **short**
  - иначе -> **HOLD** (удерживаем предыдущую позицию, `hold_keep_prev=True`)
- При смене `session_key` позиция сбрасывается в 0.
- Комиссия: `commission_rt=0.001` (0.1% round-trip), списание при смене позиции.
- Доходность: `ret_next = close_price.pct_change().shift(-1)`, PnL считается как `sum(pos * ret_next) - fee`.

## Сравниваем 5 режимов уверенности

- **20-80**: `threshold=0.80`
- **25-75**: `threshold=0.75`
- **30-70**: `threshold=0.70`
- **35-65**: `threshold=0.65`
- **40-60**: `threshold=0.60`

## Метрики

- `auc_val`, `auc_test`
- `net_%`, `trades`
- `avg_%_per_trade = net_% / trades`

---

## Почему эти модели

| Модель | Сильная сторона |
|--------|------------------|
| **LSTM** | Долгосрочные зависимости во времени |
| **GRU** | Быстрее/проще LSTM при схожем качестве |
| **1D CNN** | Локальные паттерны и высокая скорость |
| **Transformer** | Гибкие зависимости через attention |

## 1. Импорты и загрузка данных

Данные: `data_labeled_tp_sl_1_05.parquet`. Фичи: `features_selected_tp_sl_1_05.txt`. Temporal split: 8 train, 1 val, 1 test.

In [1]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Не найден {labeled_path}. Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError(f'Не найден {feat_path}. Запустите 06_Feature_Selection.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

TARGET_COL = 'target'
valid = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
if len(dates) < 10:
    raise ValueError(f'Мало дат: {len(dates)}. Нужно минимум 10 дней.')
train_dates = set(dates[:8])
val_dates = set([dates[8]])
test_dates = set([dates[9]])
train_df = valid[valid['date'].isin(train_dates)].copy()
val_df = valid[valid['date'].isin(val_dates)].copy()
test_df = valid[valid['date'].isin(test_dates)].copy()

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
train_df = train_df.sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df = val_df.sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df = test_df.sort_values(['session_key', sort_col]).reset_index(drop=True)

COMMISSION_RT = 0.001
THR_PROD = 0.60

print(f'Temporal split: train={min(train_dates)}..{max(train_dates)} ({len(train_df):,}) | val={dates[8]} ({len(val_df):,}) | test={dates[9]} ({len(test_df):,})')
print(f'Features: {len(BASE_FEATURES)}')

PyTorch: 2.5.1+cu121
Device: cuda
Temporal split: train=2026-02-01..2026-02-08 (146,693) | val=2026-02-09 (24,014) | test=2026-02-10 (15,356)
Features: 22


## 2. Функции: build_temporal_arrays, backtest, train

In [2]:
THRESHOLD_BANDS = {
    '20-80': 0.80,
    '25-75': 0.75,
    '30-70': 0.70,
    '35-65': 0.65,
    '40-60': 0.60,
}


def build_temporal_arrays(df_subset, feat_cols, seq_len):
    """Для каждой сессии: последние seq_len баров → X (seq_len, n_feat), y, ret, sess."""
    df_subset = df_subset.sort_values(['session_key', sort_col]).reset_index(drop=True)
    X_list, y_list, ret_list, sess_list = [], [], [], []
    for skey, g in df_subset.groupby('session_key', sort=False):
        g = g.dropna(subset=feat_cols + [TARGET_COL, 'ret_next']).copy()
        g = g[g[TARGET_COL].isin([-1.0, 1.0])]
        if len(g) <= seq_len:
            continue
        arr = g[feat_cols].fillna(0).values.astype(np.float32)
        for i in range(seq_len, len(g)):
            X_list.append(arr[i - seq_len:i])
            y_list.append(1 if g.iloc[i][TARGET_COL] == 1.0 else 0)
            ret_list.append(g.iloc[i]['ret_next'])
            sess_list.append(skey)
    X = np.array(X_list, dtype=np.float32) if X_list else np.empty((0, seq_len, len(feat_cols)), dtype=np.float32)
    y = np.array(y_list, dtype=np.int32) if y_list else np.array([], dtype=np.int32)
    ret = np.array(ret_list, dtype=np.float64) if ret_list else np.array([])
    sess = np.array(sess_list) if sess_list else np.array([])
    return X, y, ret, sess


def backtest_pnl(proba, ret, session_ids, threshold=THR_PROD, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    """Логика как в продовых экспериментах: buy если p>=thr, sell если p<=1-thr, иначе hold prev."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total = pos_changed.sum() * (commission_rt / 2.0)
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_per_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_per_trade}


def predict_proba(model, X, batch_size=2048):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i + batch_size]).float().to(device)
            pb = model(xb).detach().cpu().numpy().reshape(-1)
            preds.append(pb)
    return np.concatenate(preds) if preds else np.array([], dtype=np.float32)


def train_and_eval(model, X_train, y_train, X_val, y_val, X_test, y_test, epochs=80, batch_size=256):
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xt = torch.from_numpy(X_train).float()
    yt = torch.from_numpy(y_train).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    best_val_auc, best_state, patience_cnt = 0.0, None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        pv = predict_proba(model, X_val)
        auc_val = roc_auc_score(y_val, pv) if len(y_val) > 0 else 0.5
        if auc_val > best_val_auc:
            best_val_auc = auc_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= 12:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    proba_val = predict_proba(model, X_val)
    proba_test = predict_proba(model, X_test)
    auc_val = roc_auc_score(y_val, proba_val) if len(y_val) > 0 else 0.5
    auc_test = roc_auc_score(y_test, proba_test) if len(y_test) > 0 else 0.5
    return auc_val, auc_test, proba_test


def build_trade_rows(model_name, seq_len, auc_val, auc_test, proba_test, ret_test, sess_test):
    rows = []
    for band_name, thr in THRESHOLD_BANDS.items():
        bt = backtest_pnl(proba_test, ret_test, sess_test, threshold=thr)
        rows.append({
            'model': model_name,
            'seq_len': seq_len,
            'threshold_band': band_name,
            'threshold': thr,
            'auc_val': auc_val,
            'auc_test': auc_test,
            'net_%': bt['net_%'],
            'trades': bt['trades'],
            'avg_%_per_trade': bt['avg_%_per_trade']
        })
    return rows

## 3. Подготовка данных для разных длин последовательности

Подготовим данные для seq_len = 5, 15, 30, 60, 120.

In [3]:
df_temp = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
df_temp = df_temp[df_temp[TARGET_COL].isin([-1.0, 1.0])]
df_temp['ret_next'] = df_temp.groupby('session_key')['close_price'].pct_change().shift(-1)
df_temp['date'] = pd.to_datetime(df_temp['datetime'], utc=True).dt.date
df_temp = df_temp.dropna(subset=['ret_next'])

train_temp = df_temp[df_temp['date'].isin(train_dates)]
val_temp = df_temp[df_temp['date'].isin(val_dates)]
test_temp = df_temp[df_temp['date'].isin(test_dates)]

SEQ_LENS = [5, 15, 30, 60, 120, 180]
data_by_len = {}
for sl in SEQ_LENS:
    X_tr, y_tr, _, _ = build_temporal_arrays(train_temp, BASE_FEATURES, sl)
    X_va, y_va, _, _ = build_temporal_arrays(val_temp, BASE_FEATURES, sl)
    X_te, y_te, ret_te, sess_te = build_temporal_arrays(test_temp, BASE_FEATURES, sl)
    if len(X_tr) == 0 or len(X_te) == 0:
        print(f'seq_len={sl}: недостаточно данных')
        continue
    scaler = StandardScaler()
    n_f = X_tr.shape[2]
    X_tr_flat = X_tr.reshape(-1, n_f)
    scaler.fit(X_tr_flat)
    X_tr = scaler.transform(X_tr_flat).reshape(-1, sl, n_f).astype(np.float32)
    X_va_flat = X_va.reshape(-1, n_f)
    X_va = scaler.transform(X_va_flat).reshape(-1, sl, n_f).astype(np.float32)
    X_te_flat = X_te.reshape(-1, n_f)
    X_te = scaler.transform(X_te_flat).reshape(-1, sl, n_f).astype(np.float32)
    data_by_len[sl] = {'X_train': X_tr, 'y_train': y_tr, 'X_val': X_va, 'y_val': y_va,
                       'X_test': X_te, 'y_test': y_te, 'ret_test': ret_te, 'sess_test': sess_te, 'n_feat': n_f}
    print(f'seq_len={sl}: train={X_tr.shape[0]:,}, val={X_va.shape[0]:,}, test={X_te.shape[0]:,}')

seq_len=5: train=142,359, val=23,309, test=14,845
seq_len=15: train=133,886, val=21,899, test=13,843
seq_len=30: train=121,953, val=19,790, test=12,396
seq_len=60: train=101,819, val=15,761, test=9,659
seq_len=120: train=70,833, val=10,112, test=5,436
seq_len=180: train=49,312, val=6,238, test=2,873


## 4. LSTM

In [4]:
class LSTMNet(nn.Module):
    def __init__(self, n_feat, hidden=64, hidden2=32):
        super().__init__()
        self.lstm1 = nn.LSTM(n_feat, hidden, batch_first=True)
        self.lstm2 = nn.LSTM(hidden, hidden2, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(hidden2, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.drop(x)
        x, _ = self.lstm2(x)
        x = self.drop(x[:, -1, :])
        return self.fc(x)

results_lstm = []
for sl in SEQ_LENS:
    if sl not in data_by_len:
        continue
    d = data_by_len[sl]
    model = LSTMNet(d['n_feat'])
    auc_v, auc_t, proba_t = train_and_eval(model, d['X_train'], d['y_train'], d['X_val'], d['y_val'],
                                           d['X_test'], d['y_test'])
    rows = build_trade_rows(f'lstm_seq{sl}', sl, auc_v, auc_t, proba_t, d['ret_test'], d['sess_test'])
    results_lstm.extend(rows)
    for r in rows:
        print(f"LSTM seq{sl} [{r['threshold_band']}]: AUC_val={auc_v:.4f} AUC_test={auc_t:.4f} net={r['net_%']:.2f}% trades={r['trades']} avg/trade={r['avg_%_per_trade']:.4f}%")

print('LSTM готово.')

LSTM seq5 [20-80]: AUC_val=0.8062 AUC_test=0.7400 net=834.67% trades=724 avg/trade=1.1529%
LSTM seq5 [25-75]: AUC_val=0.8062 AUC_test=0.7400 net=944.01% trades=1027 avg/trade=0.9192%
LSTM seq5 [30-70]: AUC_val=0.8062 AUC_test=0.7400 net=957.84% trades=1353 avg/trade=0.7079%
LSTM seq5 [35-65]: AUC_val=0.8062 AUC_test=0.7400 net=1000.32% trades=1715 avg/trade=0.5833%
LSTM seq5 [40-60]: AUC_val=0.8062 AUC_test=0.7400 net=1053.16% trades=2057 avg/trade=0.5120%
LSTM seq15 [20-80]: AUC_val=0.8030 AUC_test=0.7503 net=691.29% trades=499 avg/trade=1.3854%
LSTM seq15 [25-75]: AUC_val=0.8030 AUC_test=0.7503 net=769.85% trades=657 avg/trade=1.1718%
LSTM seq15 [30-70]: AUC_val=0.8030 AUC_test=0.7503 net=876.37% trades=833 avg/trade=1.0521%
LSTM seq15 [35-65]: AUC_val=0.8030 AUC_test=0.7503 net=964.22% trades=1004 avg/trade=0.9604%
LSTM seq15 [40-60]: AUC_val=0.8030 AUC_test=0.7503 net=994.79% trades=1249 avg/trade=0.7965%
LSTM seq30 [20-80]: AUC_val=0.7911 AUC_test=0.7195 net=474.48% trades=341 avg

## 5. GRU

In [5]:
class GRUNet(nn.Module):
    def __init__(self, n_feat, hidden=64, hidden2=32):
        super().__init__()
        self.gru1 = nn.GRU(n_feat, hidden, batch_first=True)
        self.gru2 = nn.GRU(hidden, hidden2, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(hidden2, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        x, _ = self.gru1(x)
        x = self.drop(x)
        x, _ = self.gru2(x)
        x = self.drop(x[:, -1, :])
        return self.fc(x)

results_gru = []
for sl in SEQ_LENS:
    if sl not in data_by_len:
        continue
    d = data_by_len[sl]
    model = GRUNet(d['n_feat'])
    auc_v, auc_t, proba_t = train_and_eval(model, d['X_train'], d['y_train'], d['X_val'], d['y_val'],
                                           d['X_test'], d['y_test'])
    rows = build_trade_rows(f'gru_seq{sl}', sl, auc_v, auc_t, proba_t, d['ret_test'], d['sess_test'])
    results_gru.extend(rows)
    for r in rows:
        print(f"GRU seq{sl} [{r['threshold_band']}]: AUC_val={auc_v:.4f} AUC_test={auc_t:.4f} net={r['net_%']:.2f}% trades={r['trades']} avg/trade={r['avg_%_per_trade']:.4f}%")

print('GRU готово.')

GRU seq5 [20-80]: AUC_val=0.8068 AUC_test=0.7416 net=925.92% trades=769 avg/trade=1.2041%
GRU seq5 [25-75]: AUC_val=0.8068 AUC_test=0.7416 net=978.38% trades=1058 avg/trade=0.9247%
GRU seq5 [30-70]: AUC_val=0.8068 AUC_test=0.7416 net=965.04% trades=1376 avg/trade=0.7013%
GRU seq5 [35-65]: AUC_val=0.8068 AUC_test=0.7416 net=1056.16% trades=1667 avg/trade=0.6336%
GRU seq5 [40-60]: AUC_val=0.8068 AUC_test=0.7416 net=988.31% trades=2094 avg/trade=0.4720%
GRU seq15 [20-80]: AUC_val=0.8146 AUC_test=0.7553 net=662.07% trades=521 avg/trade=1.2708%
GRU seq15 [25-75]: AUC_val=0.8146 AUC_test=0.7553 net=763.64% trades=662 avg/trade=1.1535%
GRU seq15 [30-70]: AUC_val=0.8146 AUC_test=0.7553 net=884.67% trades=782 avg/trade=1.1313%
GRU seq15 [35-65]: AUC_val=0.8146 AUC_test=0.7553 net=909.32% trades=952 avg/trade=0.9552%
GRU seq15 [40-60]: AUC_val=0.8146 AUC_test=0.7553 net=1011.92% trades=1174 avg/trade=0.8619%
GRU seq30 [20-80]: AUC_val=0.8013 AUC_test=0.7320 net=432.12% trades=370 avg/trade=1.167

## 6. 1D CNN

In [6]:
class CNN1DNet(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.conv1 = nn.Conv1d(n_feat, 64, 5, padding=2)
        self.conv2 = nn.Conv1d(64, 32, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.max_pool1d(x, 2)
        x = nn.functional.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        x = self.drop(x)
        return self.fc(x)

results_cnn = []
for sl in SEQ_LENS:
    if sl not in data_by_len:
        continue
    d = data_by_len[sl]
    model = CNN1DNet(d['n_feat'])
    auc_v, auc_t, proba_t = train_and_eval(model, d['X_train'], d['y_train'], d['X_val'], d['y_val'],
                                           d['X_test'], d['y_test'])
    rows = build_trade_rows(f'cnn1d_seq{sl}', sl, auc_v, auc_t, proba_t, d['ret_test'], d['sess_test'])
    results_cnn.extend(rows)
    for r in rows:
        print(f"1D CNN seq{sl} [{r['threshold_band']}]: AUC_val={auc_v:.4f} AUC_test={auc_t:.4f} net={r['net_%']:.2f}% trades={r['trades']} avg/trade={r['avg_%_per_trade']:.4f}%")

print('1D CNN готово.')

1D CNN seq5 [20-80]: AUC_val=0.8100 AUC_test=0.7429 net=880.04% trades=796 avg/trade=1.1056%
1D CNN seq5 [25-75]: AUC_val=0.8100 AUC_test=0.7429 net=958.03% trades=1107 avg/trade=0.8654%
1D CNN seq5 [30-70]: AUC_val=0.8100 AUC_test=0.7429 net=1020.59% trades=1436 avg/trade=0.7107%
1D CNN seq5 [35-65]: AUC_val=0.8100 AUC_test=0.7429 net=990.78% trades=1778 avg/trade=0.5572%
1D CNN seq5 [40-60]: AUC_val=0.8100 AUC_test=0.7429 net=1117.42% trades=2209 avg/trade=0.5059%
1D CNN seq15 [20-80]: AUC_val=0.8125 AUC_test=0.7483 net=603.84% trades=494 avg/trade=1.2223%
1D CNN seq15 [25-75]: AUC_val=0.8125 AUC_test=0.7483 net=769.15% trades=695 avg/trade=1.1067%
1D CNN seq15 [30-70]: AUC_val=0.8125 AUC_test=0.7483 net=816.36% trades=936 avg/trade=0.8722%
1D CNN seq15 [35-65]: AUC_val=0.8125 AUC_test=0.7483 net=1000.17% trades=1211 avg/trade=0.8259%
1D CNN seq15 [40-60]: AUC_val=0.8125 AUC_test=0.7483 net=1021.11% trades=1567 avg/trade=0.6516%
1D CNN seq30 [20-80]: AUC_val=0.7887 AUC_test=0.7133 ne

## 7. Transformer (self-attention)

Простейший Transformer: positional encoding + один блок MultiHeadAttention + Dense. Attention позволяет каждому бару учитывать все остальные.

In [7]:
class TransformerNet(nn.Module):
    def __init__(self, seq_len, n_feat, d_model=32, num_heads=4, ff_dim=32):
        super().__init__()
        self.seq_len = seq_len
        self.proj = nn.Linear(n_feat, d_model)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Dropout(0.2), nn.Linear(ff_dim, 1), nn.Sigmoid())
    def forward(self, x):
        B = x.size(0)
        x = self.proj(x)
        pos = torch.arange(self.seq_len, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_embed(pos)
        x, _ = self.attn(x, x, x)
        x = self.drop(x)
        x = x.mean(dim=1)
        return self.fc(x)

results_transformer = []
for sl in SEQ_LENS:
    if sl not in data_by_len:
        continue
    d = data_by_len[sl]
    model = TransformerNet(sl, d['n_feat'])
    auc_v, auc_t, proba_t = train_and_eval(model, d['X_train'], d['y_train'], d['X_val'], d['y_val'],
                                           d['X_test'], d['y_test'])
    rows = build_trade_rows(f'transformer_seq{sl}', sl, auc_v, auc_t, proba_t, d['ret_test'], d['sess_test'])
    results_transformer.extend(rows)
    for r in rows:
        print(f"Transformer seq{sl} [{r['threshold_band']}]: AUC_val={auc_v:.4f} AUC_test={auc_t:.4f} net={r['net_%']:.2f}% trades={r['trades']} avg/trade={r['avg_%_per_trade']:.4f}%")

print('Transformer готово.')

Transformer seq5 [20-80]: AUC_val=0.7932 AUC_test=0.7158 net=752.92% trades=519 avg/trade=1.4507%
Transformer seq5 [25-75]: AUC_val=0.7932 AUC_test=0.7158 net=721.93% trades=862 avg/trade=0.8375%
Transformer seq5 [30-70]: AUC_val=0.7932 AUC_test=0.7158 net=901.35% trades=1224 avg/trade=0.7364%
Transformer seq5 [35-65]: AUC_val=0.7932 AUC_test=0.7158 net=1051.12% trades=1632 avg/trade=0.6441%
Transformer seq5 [40-60]: AUC_val=0.7932 AUC_test=0.7158 net=979.49% trades=2145 avg/trade=0.4566%
Transformer seq15 [20-80]: AUC_val=0.8060 AUC_test=0.7206 net=401.29% trades=358 avg/trade=1.1209%
Transformer seq15 [25-75]: AUC_val=0.8060 AUC_test=0.7206 net=517.70% trades=474 avg/trade=1.0922%
Transformer seq15 [30-70]: AUC_val=0.8060 AUC_test=0.7206 net=700.63% trades=676 avg/trade=1.0364%
Transformer seq15 [35-65]: AUC_val=0.8060 AUC_test=0.7206 net=854.86% trades=905 avg/trade=0.9446%
Transformer seq15 [40-60]: AUC_val=0.8060 AUC_test=0.7206 net=810.42% trades=1228 avg/trade=0.6600%
Transforme

## 8. Итоговая таблица (30-70 vs 40-60)

In [8]:
all_results = results_lstm + results_gru + results_cnn + results_transformer
summary = pd.DataFrame(all_results)[['model', 'seq_len', 'threshold_band', 'threshold', 'auc_val', 'auc_test', 'net_%', 'trades', 'avg_%_per_trade']]
summary = summary.sort_values('net_%', ascending=False).reset_index(drop=True)
print('Сравнение глубоких сетей (20-80, 25-75, 30-70, 35-65, 40-60):')
print(summary.to_string(index=False))
print()
print('Лучший по net PnL:', summary.iloc[0]['model'], f"seq={int(summary.iloc[0]['seq_len'])}", f"band={summary.iloc[0]['threshold_band']}", f"({summary.iloc[0]['net_%']:.2f}%)")
print('(Только эксперимент — ничего не сохранено в прод)')

Сравнение глубоких сетей (20-80, 25-75, 30-70, 35-65, 40-60):
             model  seq_len threshold_band  threshold  auc_val  auc_test       net_%  trades  avg_%_per_trade
        cnn1d_seq5        5          40-60       0.60 0.810009  0.742871 1117.422745    2209         0.505850
          gru_seq5        5          35-65       0.65 0.806797  0.741568 1056.158785    1667         0.633569
         lstm_seq5        5          40-60       0.60 0.806235  0.740017 1053.158796    2057         0.511988
  transformer_seq5        5          35-65       0.65 0.793223  0.715799 1051.115963    1632         0.644066
       cnn1d_seq15       15          40-60       0.60 0.812494  0.748256 1021.113304    1567         0.651636
        cnn1d_seq5        5          30-70       0.70 0.810009  0.742871 1020.592834    1436         0.710719
         gru_seq15       15          40-60       0.60 0.814562  0.755294 1011.919378    1174         0.861942
         lstm_seq5        5          35-65       0.65 0.80

## Итог и выводы

### Условия эксперимента
- Модели: **LSTM, GRU, 1D CNN, Transformer** (PyTorch, GPU CUDA).
- Длины последовательностей: **5, 15, 30, 60, 120, 180** баров.
- 5 режимов уверенности: **20-80, 25-75, 30-70, 35-65, 40-60**.
- Логика сделок: `p >= thr` → long, `p <= 1−thr` → short, иначе hold (удерживаем предыдущую позицию).
- **Комиссия учтена**: `commission_rt = 0.001` (0.1% round-trip) списывается при каждой смене позиции. Все значения `net_%` уже после вычета комиссии.
- **Не учтён slippage** (проскальзывание): в реальной торговле на Bybit фьючерсах рыночные ордера дают ~0.02–0.1% дополнительных потерь на вход+выход.

### Контекст: что важно для реальной торговли

Чистый `net_%` — не главный критерий для прода. Ключевые ограничения:

1. **Количество сделок**: ~2000 сделок за 1 тестовый день — чрезмерно. Каждая сделка несёт slippage, который бектест не моделирует. Реалистичный диапазон: **300–700 сделок/день**.
2. **avg_%_per_trade**: при slippage ~0.05% на сделку, модель с `avg/trade < 0.5%` может стать убыточной в реале. Целевой порог: **avg/trade ≥ 0.8%**.
3. **Переобучение**: разрыв AUC_val vs AUC_test (0.81 → 0.75) сигнализирует о частичном overfitting. В проде результаты будут ближе к тестовым.
4. **Оценка реальной прибыли**: из бектестовых net_% реалистично ожидать **30–60%** после slippage, задержек, частичного исполнения.

### Лучшие кандидаты (баланс точности, прибыли на сделку, количества сделок)

| Модель | seq | band | AUC_test | net_% | trades | avg/trade | Комментарий |
|--------|-----|------|----------|-------|--------|-----------|-------------|
| **gru_seq15** | 15 | 30-70 | 0.755 | 885% | 782 | **1.13%** | Лучший баланс: достаточно сделок, высокий avg/trade |
| **lstm_seq15** | 15 | 25-75 | 0.750 | 770% | 657 | **1.17%** | Меньше сделок, стабильнее |
| **lstm_seq15** | 15 | 30-70 | 0.750 | 876% | 833 | **1.05%** | Чуть больше сделок, всё ещё хороший avg |
| **gru_seq15** | 15 | 25-75 | 0.755 | 764% | 662 | **1.15%** | Консервативный вариант GRU |
| **lstm_seq15** | 15 | 20-80 | 0.750 | 691% | 499 | **1.39%** | Самый жирный avg/trade, но мало сделок |
| **gru_seq60** | 60 | 20-80 | 0.715 | 271% | 223 | **1.22%** | Ультраконсервативно, максимальный запас на slippage |

### Паттерны

1. **seq_len = 15 — оптимум.** Лучший баланс AUC, net_%, trades и avg/trade по всем моделям. Seq_len=5 генерирует больше net_%, но за счёт >1500 сделок — в реале slippage съедает преимущество.
2. **GRU ≥ LSTM.** При одинаковых параметрах GRU даёт чуть выше AUC_test и стабильнее avg/trade. При этом GRU проще и быстрее в обучении.
3. **1D CNN — много сделок, низкий avg/trade.** CNN хорошо ловит локальные паттерны, но генерирует слишком частые сигналы — неэффективно для прода с учётом slippage.
4. **Transformer — слабее остальных.** На наших данных (22 фичи, короткие окна) attention не даёт преимущества. Вероятно, данных недостаточно для обучения attention-весов.
5. **seq_len ≥ 120 — не работает.** AUC падает ниже 0.66, avg/trade < 0.5%. Длинные последовательности теряют релевантность для минутных баров.
6. **Порог 25-75 и 30-70 — оптимальная зона.** 20-80 слишком консервативен (мало сделок — нестабильная статистика). 40-60 слишком агрессивен (>1500 сделок, avg/trade < 0.6%).

### Сравнение с классическими моделями (ноутбук 13)

В `13_Complex_Models_Ensemble_And_Entry_Exit_Logic.ipynb` лучший результат показал **lgb_seq** (LightGBM + sequence features): `AUC_test=0.629, net=1035%, trades=1564, avg/trade≈0.66%`.

Глубокие сети (GRU/LSTM seq15) при сопоставимом net_% дают **существенно выше AUC_test** (0.75 vs 0.63) и **выше avg/trade** (1.0–1.4% vs 0.66%). Это означает, что нейросети лучше различают направление и делают более точные входы. Однако LightGBM работает в 100× быстрее и не требует GPU.

### Рекомендация

Для прода — рассмотреть **gru_seq15 [25-75 или 30-70]** как основную модель входа. Комбинация высокого AUC, умеренного числа сделок и avg/trade > 1% даёт наибольший запас прочности после slippage и реальных условий биржи.